# LOGIVISION — Train YOLOv8 on Google Colab (T4)

**Purpose.** Fine-tune YOLOv8n on the **LOCO** real-warehouse dataset
using a free Colab T4 GPU. 50 epochs at 640 px takes **~25 minutes on
T4**, vs hours on an M3 CPU.

## Before you run

1. *Runtime → Change runtime type → T4 GPU* (confirm top-right).
2. That's it — **no credentials needed**. LOCO is public-domain (CC0),
   so the download cell needs no Kaggle/login setup.

## What this notebook does

Trains on **LOCO** (Logistics Objects in Context) — real photos from five
operating warehouse environments. Classes: `small_load_carrier, forklift,
pallet, stillage, pallet_truck`.

| Cell | Step | Time on T4 |
|---|---|---|
| 1 | Detect Colab; clone repo (depth=1) | 5 s |
| 2 | Install ultralytics, pyyaml, mlflow | 30 s |
| 3 | (no-op — LOCO needs no credentials) | <1 s |
| 4 | Download LOCO (~769 MB, CC0) | 1-3 min |
| 5 | Convert COCO → YOLO, scene-separated split (`scripts/prepare_loco.py`) | 20 s |
| 6 | Train YOLOv8n: 50 epochs, imgsz=640, batch=32, AdamW, cos_lr | **~25 min** |
| 7 | Validate on the held-out test split (subset 4 — unseen warehouse) | 30 s |
| 8 | Plot training curves | 5 s |
| 9 | Download `best.pt` + `results.csv` to your machine | 5 s |

At the end you drop the downloaded files into your local repo and
register the model in MLflow with `make register-from-colab`.

In [ ]:
# 1. Detect Colab and clone the repo (shallow).
import os, sys, subprocess, pathlib

IN_COLAB = 'google.colab' in sys.modules
print(f'Running on Colab: {IN_COLAB}')
if IN_COLAB:
    if not pathlib.Path('logivision_v2').is_dir():
        subprocess.run(['git', 'clone', '--depth=1',
                        'https://github.com/Ayalem/logivision_v2.git'], check=True)
    os.chdir('logivision_v2')
    print('cwd:', pathlib.Path.cwd())

# Critical: put the repo root + scripts/ on sys.path so `import services.*`
# resolves to the cloned files. Without this Python only searches
# site-packages — `pip install services` would install an UNRELATED PyPI
# package; do not do that.
REPO = pathlib.Path.cwd().resolve()
for p in (str(REPO), str(REPO / 'scripts')):
    if p not in sys.path:
        sys.path.insert(0, p)
print('sys.path[0:3] =', sys.path[:3])

# Sanity: confirm we can see the LOCO fetch script + the services pkg.
assert pathlib.Path('scripts/fetch_loco.py').is_file(), \
    'fetch_loco.py missing - did the repo clone correctly?'
assert pathlib.Path('services/model_server/service.py').is_file(), \
    'services/model_server/service.py missing - your clone is stale; re-run git clone'

# GPU check
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA: {torch.version.cuda}')


In [ ]:
# 2. Install deps. Colab pre-imports numpy; installing ultralytics can
#    swap numpy under already-loaded C extensions (torch/opencv), causing the
#    "numpy.dtype size changed" ABI error later. Fix: install, then restart
#    the runtime ONCE so every binary re-links against a single numpy. The
#    guard file means the restart only fires on the first run — after it
#    restarts, just do Runtime -> Run all again.
%pip install -q ultralytics pyyaml mlflow
import os, sys, pathlib
if 'google.colab' in sys.modules:
    _flag = pathlib.Path('/content/.logivision_deps_ok')
    if not _flag.exists():
        _flag.touch()
        print('Deps installed — restarting runtime once to settle numpy ABI.')
        print('When it comes back, run Runtime -> Run all again.')
        os.kill(os.getpid(), 9)
print('deps ready')


In [ ]:
# 3. LOCO is public-domain (CC0) — no credentials required.
#    (This cell is intentionally a no-op; kept so cell numbering matches
#    the guide and any older instructions.)
print('LOCO is CC0 public domain — no Kaggle/login needed.')


In [ ]:
# 4. Download LOCO — real warehouse imagery (~769 MB, CC0, no login).
#    5,593 photos from 5 operating logistics environments.
import subprocess, sys
subprocess.run([sys.executable, 'scripts/fetch_loco.py'], check=True)
print('LOCO downloaded + extracted under datasets/raw/loco/')


In [ ]:
# 5. Convert LOCO (COCO) -> YOLO with a SCENE-SEPARATED split.
#
#    Each LOCO subset is a distinct warehouse environment, so splitting by
#    subset (train = 2,3,5 / val = 1 / test = 4) guarantees no scene leaks
#    across splits — the honest, leak-free evaluation setup. Classes:
#    small_load_carrier, forklift, pallet, stillage, pallet_truck.
import subprocess, sys, pathlib
subprocess.run([sys.executable, 'scripts/prepare_loco.py'], check=True)

DATA_YAML = pathlib.Path('datasets/processed/loco/data.yaml').resolve()
print('LOCO data.yaml:', DATA_YAML)
print(DATA_YAML.read_text())


In [ ]:
# 6. Train. ~25 min on T4.
#    Hyperparameters chosen for the soutenance demo:
#      - 50 epochs (good convergence on 2,820 LOCO train images)
#      - imgsz=640 (production size; not the 320 demo size)
#      - batch=32 (T4 has 16 GB - safe headroom; bump to 48 if you want)
#      - cos_lr=True (anneals smoothly; matches ULMFiT recipe)
#      - patience=15 (early stop if val mAP plateaus)
#      - device=0 (GPU 0)

# Defensive: if cells 1-5 weren't run in order, recover DATA_YAML from disk.
# This lets you re-run cell 6 alone after a restart, as long as cell 5 ran
# at least once in the current Colab session.
try:
    DATA_YAML
except NameError:
    import pathlib
    # Recover the scene-separated LOCO yaml written by cell 5.
    DATA_YAML = pathlib.Path('datasets/processed/loco/data.yaml').resolve()
    if not DATA_YAML.is_file():
        raise FileNotFoundError(
            f'{DATA_YAML} not found — please run cells 1-5 (Runtime -> Run all).'
        )
    print(f'(recovered DATA_YAML from disk: {DATA_YAML})')

from ultralytics import YOLO

model = YOLO('yolov8n.pt')                  # COCO weights as starting point
results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=32,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    patience=15,
    device=0,
    project='runs',
    name='colab_loco_50ep',
    exist_ok=True,
    verbose=False,
    plots=True,
    seed=42,
)
print('training done')


In [ ]:
# 7. Final validation on the held-out test split (not the val we early-stopped on).
val_results = model.val(data=str(DATA_YAML), split='test', verbose=False)
print('--- Final test-set metrics ---')
print(f'mAP@0.5         : {val_results.box.map50:.4f}')
print(f'mAP@0.5:0.95    : {val_results.box.map:.4f}')
print(f'mean precision  : {val_results.box.mp:.4f}')
print(f'mean recall     : {val_results.box.mr:.4f}')
print('per class:')
for i, name in val_results.names.items():
    print(f'  {name:<14} mAP50={val_results.box.maps[i]:.4f}')


In [ ]:
# 8. Plot the training curves so they're embedded in the notebook output.
import pandas as pd
import matplotlib.pyplot as plt

candidates = sorted(pathlib.Path('runs').glob('**/colab_loco_50ep'))
run_dir = next((p for p in candidates if (p / 'results.csv').is_file()), None)
if run_dir is None:
    run_dir = next((p for p in candidates if (p / 'weights' / 'best.pt').is_file()), None)
if run_dir is None:
    raise FileNotFoundError(
        'No run directory containing results.csv or weights/best.pt under runs/. '
        'Did cell 6 finish?  Candidates: ' + str(candidates)
    )
print('using run_dir =', run_dir)
df = pd.read_csv(run_dir / 'results.csv')
df.columns = [c.strip() for c in df.columns]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(df['epoch'], df['metrics/mAP50(B)'],     label='mAP@0.5',     color='#2563EB', linewidth=2)
axes[0].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95', color='#06B6D4', linewidth=2)
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('mAP'); axes[0].set_title('Validation mAP')
axes[0].grid(alpha=0.3); axes[0].legend()

axes[1].plot(df['epoch'], df['train/box_loss'], label='box loss', color='#EF4444')
axes[1].plot(df['epoch'], df['train/cls_loss'], label='cls loss', color='#F59E0B')
axes[1].plot(df['epoch'], df['train/dfl_loss'], label='dfl loss', color='#8B5CF6')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('loss'); axes[1].set_title('Training losses')
axes[1].grid(alpha=0.3); axes[1].legend()
plt.tight_layout(); plt.show()

best = df.loc[df['metrics/mAP50(B)'].idxmax()]
print(f'Best epoch: {int(best.epoch)} -> mAP50={best["metrics/mAP50(B)"]:.3f}, '
      f'precision={best["metrics/precision(B)"]:.3f}, recall={best["metrics/recall(B)"]:.3f}')


In [ ]:
# 9. Package best.pt + last.pt + results.csv + confusion matrix into a
#    single zip and trigger a Colab download to your machine.
import shutil
from datetime import datetime

candidates = sorted(pathlib.Path('runs').glob('**/colab_loco_50ep'))
run_dir = next((p for p in candidates if (p / 'results.csv').is_file()), None)
if run_dir is None:
    run_dir = next((p for p in candidates if (p / 'weights' / 'best.pt').is_file()), None)
if run_dir is None:
    raise FileNotFoundError(
        'No run directory containing results.csv or weights/best.pt under runs/. '
        'Did cell 6 finish?  Candidates: ' + str(candidates)
    )
print('using run_dir =', run_dir)
stamp = datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
bundle = pathlib.Path(f'/content/logivision_colab_run_{stamp}')
bundle.mkdir(exist_ok=True)
for f in ('weights/best.pt', 'weights/last.pt', 'results.csv',
          'confusion_matrix.png', 'confusion_matrix_normalized.png',
          'args.yaml', 'labels.jpg'):
    src = run_dir / f
    if src.is_file():
        shutil.copy(src, bundle / pathlib.Path(f).name)
zip_path = shutil.make_archive(str(bundle), 'zip', bundle)
print(f'bundle: {zip_path} ({pathlib.Path(zip_path).stat().st_size/1e6:.1f} MB)')

# Trigger Colab download (or use right-click in the file panel if this fails)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)


## Next steps on your laptop

After the download completes:

```bash
# 1. Unzip into the repo's ml/runs directory under a colab-named subdir.
RUN_NAME=$(basename ~/Downloads/logivision_colab_run_*.zip .zip)
mkdir -p ml/runs/$RUN_NAME/weights
unzip -o ~/Downloads/${RUN_NAME}.zip -d ml/runs/$RUN_NAME/
mv ml/runs/$RUN_NAME/best.pt ml/runs/$RUN_NAME/weights/best.pt
mv ml/runs/$RUN_NAME/last.pt ml/runs/$RUN_NAME/weights/last.pt

# 2. Register in your local MLflow + promote to Production.
make register-from-colab RUN=$RUN_NAME

# 3. Restart the inference worker so it picks up the new model.
make worker-restart
```

The dashboard's `System` tab will show the new model version + metrics on
next refresh. Detections start using the Colab-trained weights in ~10 s.

## Troubleshooting

| Symptom | Fix |
|---|---|
| LOCO download fails (cell 4) | Re-run the cell; the TUM link is occasionally slow. Check Colab has internet. |
| `CUDA out of memory` | Drop `batch=32` to `batch=16` in cell 6 |
| Training stalls at 0 mAP after 5+ epochs | Re-check `data.yaml` — classes/paths must match what cell 5 wrote |
| `files.download()` does nothing | Click the file panel on the left, right-click the zip, "Download" |
